In [31]:
# Housing Price Prediction - Model Comparison
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, accuracy_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [32]:
df = pd.read_csv("website/cairo Real-Estate-revised.csv")

numerical_columns = ["Area", "Bedrooms", "Bathrooms", "Floor"]

for col in numerical_columns:
    df[col] = df[col].fillna(df[col].median())

df = df.drop(columns=["ID", "City"], errors="ignore")

numerical_columns = [
    "Area",
    "Bedrooms",
    "Bathrooms",
    "Floor",
    "Price"
]

for col in numerical_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df = df[
        (df[col] >= lower_bound) &
        (df[col] <= upper_bound)
    ]

for col in [
    "Location",
    "View",
    "Payment",
    "Finishing",
    "Furnished",
    "Parking",
    "Security"
]:
    df[col] = df[col].astype("string").str.strip()

df["Payment"] = df["Payment"].str.lower()
df["Finishing"] = df["Finishing"].str.lower()
df["Furnished"] = df["Furnished"].str.lower()
df["Parking"] = df["Parking"].str.lower()
df["Security"] = df["Security"].str.lower()

numerical_columns = [
    "Area",
    "Bedrooms",
    "Bathrooms",
    "Floor",
    "YearBuilt"
]

for col in numerical_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

# Encode binary columns
df["Finishing"] = df["Finishing"].map({
    "finished": 1,
    "unfinished": 0
})

df["Payment"] = df["Payment"].map({
    "cash": 1,
    "installments": 0
})

df["Furnished"] = df["Furnished"].map({
    "furnished": 1,
    "unfurnished": 0
})

df["Parking"] = df["Parking"].map({
    "yes": 1,
    "no": 0
})

df["Security"] = df["Security"].map({
    "yes": 1,
    "no": 0
})

label_encoder = LabelEncoder()
df["Color"] = label_encoder.fit_transform(df["Color"]) + 1

label_encoder = LabelEncoder()
df["Seller"] = label_encoder.fit_transform(df["Seller"]) + 1

df = pd.get_dummies(df, columns=["View"], dtype=int)
df = pd.get_dummies(df, columns=["Location"], dtype=int)
df = df.dropna(subset=["Price"])

X = df.drop(columns=["Price"])
y = df["Price"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [33]:
# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [34]:
# 1. Linear Regression
start = time.time()
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)
train_time_linear = time.time() - start
start = time.time()
y_pred_linear = linear_model.predict(X_test_scaled)
prediction_time_linear = time.time() - start
r2_linear = r2_score(y_test, y_pred_linear)
mse_linear = mean_squared_error(y_test, y_pred_linear)
rmse_linear = np.sqrt(mse_linear)
mae_linear = mean_absolute_error(y_test, y_pred_linear)


In [35]:
# 2. Polynomial Regression
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)
start = time.time()
poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)
train_time_poly = time.time() - start
start = time.time()
y_pred_poly = poly_model.predict(X_test_poly)
prediction_time_poly = time.time() - start
r2_poly = r2_score(y_test, y_pred_poly)
mse_poly = mean_squared_error(y_test, y_pred_poly)
rmse_poly = np.sqrt(mse_poly)
mae_poly = mean_absolute_error(y_test, y_pred_poly)


In [36]:
# 3. Ridge
start = time.time()
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)
train_time_ridge = time.time() - start
start = time.time()
y_pred_ridge = ridge_model.predict(X_test_scaled)
prediction_time_ridge = time.time() - start
r2_ridge = r2_score(y_test, y_pred_ridge)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mse_ridge)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)


In [37]:
# 4. Lasso
start = time.time()
lasso_model = Lasso(alpha=0.1, max_iter=10000)
lasso_model.fit(X_train_scaled, y_train)
train_time_lasso = time.time() - start
start = time.time()
y_pred_lasso = lasso_model.predict(X_test_scaled)
prediction_time_lasso = time.time() - start
r2_lasso = r2_score(y_test, y_pred_lasso)
mse_lasso = mean_squared_error(y_test, y_pred_lasso)
rmse_lasso = np.sqrt(mse_lasso)
mae_lasso = mean_absolute_error(y_test, y_pred_lasso)


c:\Users\slim\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.671461e+13, tolerance: 2.717e+12
  model = cd_fast.enet_coordinate_descent(


In [38]:
# 5. Elastic Net
start = time.time()
elastic_model = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
elastic_model.fit(X_train_scaled, y_train)
train_time_elastic = time.time() - start
start = time.time()
y_pred_elastic = elastic_model.predict(X_test_scaled)
prediction_time_elastic = time.time() - start
r2_elastic = r2_score(y_test, y_pred_elastic)
mse_elastic = mean_squared_error(y_test, y_pred_elastic)
rmse_elastic = np.sqrt(mse_elastic)
mae_elastic = mean_absolute_error(y_test, y_pred_elastic)


In [39]:
# 6. Decision Tree
start = time.time()
tree_model = DecisionTreeRegressor(random_state=42)
tree_model.fit(X_train, y_train)
train_time_tree = time.time() - start
start = time.time()
y_pred_tree = tree_model.predict(X_test)
prediction_time_tree = time.time() - start
r2_tree = r2_score(y_test, y_pred_tree)
mse_tree = mean_squared_error(y_test, y_pred_tree)
rmse_tree = np.sqrt(mse_tree)
mae_tree = mean_absolute_error(y_test, y_pred_tree)


In [40]:
# 7. Random Forest
start = time.time()
forest_model = RandomForestRegressor(
n_estimators=100,
random_state=42
)
forest_model.fit(X_train, y_train)
train_time_forest = time.time() - start
start = time.time()
y_pred_forest = forest_model.predict(X_test)
prediction_time_forest = time.time() - start
r2_forest = r2_score(y_test, y_pred_forest)
mse_forest = mean_squared_error(y_test, y_pred_forest)
rmse_forest = np.sqrt(mse_forest)
mae_forest = mean_absolute_error(y_test, y_pred_forest)

In [41]:
# 8. Gradient Boosting
start = time.time()
gradient_model = GradientBoostingRegressor(random_state=42)
gradient_model.fit(X_train, y_train)
train_time_gradient = time.time() - start
start = time.time()
y_pred_gradient = gradient_model.predict(X_test)
prediction_time_gradient = time.time() - start
r2_gradient = r2_score(y_test, y_pred_gradient)
mse_gradient = mean_squared_error(y_test, y_pred_gradient)
rmse_gradient = np.sqrt(mse_gradient)
mae_gradient = mean_absolute_error(y_test, y_pred_gradient)


In [42]:
# 9. XGBoost
start = time.time()
xgb_model = XGBRegressor(
n_estimators=100,
random_state=42,
objective='reg:squarederror'
)
xgb_model.fit(X_train, y_train)
train_time_xgb = time.time() - start
start = time.time()
y_pred_xgb = xgb_model.predict(X_test)
prediction_time_xgb = time.time() - start
r2_xgb = r2_score(y_test, y_pred_xgb)
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mse_xgb)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)


In [43]:
# 10. LightGBM
start = time.time()
lightgbm_model = LGBMRegressor(
n_estimators=100,
random_state=42,
verbosity=-1
)
lightgbm_model.fit(X_train, y_train)
train_time_lightgbm = time.time() - start
start = time.time()
y_pred_lightgbm = lightgbm_model.predict(X_test)
prediction_time_lightgbm = time.time() - start
r2_lightgbm = r2_score(y_test, y_pred_lightgbm)
mse_lightgbm = mean_squared_error(y_test, y_pred_lightgbm)
rmse_lightgbm = np.sqrt(mse_lightgbm)
mae_lightgbm = mean_absolute_error(y_test, y_pred_lightgbm)


In [44]:
# 11. CatBoost
start = time.time()
catboost_model = CatBoostRegressor(
iterations=100,
random_seed=42,
verbose=0
)
catboost_model.fit(X_train, y_train)
train_time_catboost = time.time() - start
start = time.time()
y_pred_catboost = catboost_model.predict(X_test)
prediction_time_catboost = time.time() - start
r2_catboost = r2_score(y_test, y_pred_catboost)
mse_catboost = mean_squared_error(y_test, y_pred_catboost)
rmse_catboost = np.sqrt(mse_catboost)
mae_catboost = mean_absolute_error(y_test, y_pred_catboost)


In [45]:
# Results
results = pd.DataFrame({
'Model': [
'Linear Regression',
'Polynomial Regression',
'Ridge',
'Lasso',
'Elastic Net',
'Decision Tree',
'Random Forest',
'Gradient Boosting',
'XGBoost',
'LightGBM',
'CatBoost'
],
'R2': [
r2_linear, r2_poly, r2_ridge, r2_lasso, r2_elastic,
r2_tree, r2_forest, r2_gradient, r2_xgb,
r2_lightgbm, r2_catboost
],
'MSE': [
mse_linear, mse_poly, mse_ridge, mse_lasso, mse_elastic,
mse_tree, mse_forest, mse_gradient, mse_xgb,
mse_lightgbm, mse_catboost
],
'RMSE': [
rmse_linear, rmse_poly, rmse_ridge, rmse_lasso, rmse_elastic,
rmse_tree, rmse_forest, rmse_gradient, rmse_xgb,
rmse_lightgbm, rmse_catboost
],
'MAE': [
mae_linear, mae_poly, mae_ridge, mae_lasso, mae_elastic,
mae_tree, mae_forest, mae_gradient, mae_xgb,
mae_lightgbm, mae_catboost
],
'Train Time': [
train_time_linear, train_time_poly, train_time_ridge,
train_time_lasso, train_time_elastic, train_time_tree,
train_time_forest, train_time_gradient, train_time_xgb,
train_time_lightgbm, train_time_catboost
],
'Prediction Time': [
prediction_time_linear, prediction_time_poly,
prediction_time_ridge, prediction_time_lasso,
prediction_time_elastic, prediction_time_tree,
prediction_time_forest, prediction_time_gradient,
prediction_time_xgb, prediction_time_lightgbm,
prediction_time_catboost
]
})
print(results.to_string(index=False))


                Model       R2          MSE         RMSE          MAE  Train Time  Prediction Time
    Linear Regression 0.731112 2.205582e+12 1.485120e+06 1.126357e+06    0.055331         0.003963
Polynomial Regression 0.729102 2.222069e+12 1.490660e+06 1.118318e+06    0.208890         0.002013
                Ridge 0.731122 2.205504e+12 1.485094e+06 1.126296e+06    0.014549         0.001005
                Lasso 0.731112 2.205582e+12 1.485120e+06 1.126357e+06    0.704162         0.000597
          Elastic Net 0.730695 2.209003e+12 1.486272e+06 1.120391e+06    0.014332         0.000507
        Decision Tree 0.491297 4.172686e+12 2.042715e+06 1.413235e+06    0.075006         0.009858
        Random Forest 0.751202 2.040792e+12 1.428563e+06 1.022222e+06    2.233722         0.042534
    Gradient Boosting 0.752159 2.032942e+12 1.425813e+06 1.050050e+06    0.639850         0.495938
              XGBoost 0.750408 2.047304e+12 1.430840e+06 1.033594e+06    0.567867         0.020046
          

In [46]:
# Best R2
print("\nBest R2:")
print(results.loc[results['R2'].idxmax(), ['Model', 'R2']])
# Lowest MSE
print("\nLowest MSE:")
print(results.loc[results['MSE'].idxmin(), ['Model', 'MSE']])
# Lowest RMSE
print("\nLowest RMSE:")
print(results.loc[results['RMSE'].idxmin(), ['Model', 'RMSE']])
# Lowest MAE
print("\nLowest MAE:")
print(results.loc[results['MAE'].idxmin(), ['Model', 'MAE']])
# Fastest Training
print("\nFastest Training:")
print(results.loc[results['Train Time'].idxmin(),
['Model', 'Train Time']])
# Fastest Prediction
print("\nFastest Prediction:")
print(results.loc[results['Prediction Time'].idxmin(),
['Model', 'Prediction Time']])


Best R2:
Model    LightGBM
R2       0.761673
Name: 9, dtype: object

Lowest MSE:
Model               LightGBM
MSE      1954900361492.69751
Name: 9, dtype: object

Lowest RMSE:
Model          LightGBM
RMSE     1398177.514299
Name: 9, dtype: object

Lowest MAE:
Model     Random Forest
MAE      1022221.548312
Name: 6, dtype: object

Fastest Training:
Model         Elastic Net
Train Time       0.014332
Name: 4, dtype: object

Fastest Prediction:
Model              Elastic Net
Prediction Time       0.000507
Name: 4, dtype: object
